# Broadcast Join Example with Ray

This notebook demonstrates a broadcast join of two pandas tables using Ray for parallel processing.

## 1. Import Libraries and Connect to Ray Cluster

In [1]:
import ray
import pandas as pd

# Connect to Ray cluster (assumes cluster is already running)
print("Connecting to Ray cluster...")
ray.init(address='auto')

print(f"Connected to Ray cluster with {ray.available_resources()['CPU']} CPUs")

2026-08-26 22:41:48,270	INFO worker.py:1833 -- Connecting to existing Ray cluster at address: 192.168.86.116:6379...
2026-08-26 22:41:48,286	INFO worker.py:2024 -- Connected to Ray cluster.


Connecting to Ray cluster...
Connected to Ray cluster with 4.0 CPUs


## 2. Create States Table (Small Table)

In [2]:
states = pd.DataFrame({
    'state_name': ['California', 'Texas', 'New York', 'Florida'],
    'state_id': [1, 2, 3, 4]
})

print(f"States table shape: {states.shape}")
states

States table shape: (4, 2)


,state_name,state_id
0,California,1
1,Texas,2
2,New York,3
3,Florida,4


## 3. Create Cities Table (Larger Table)

In [9]:
cities = pd.DataFrame({
    'city_name': [
        'Los Angeles', 'San Francisco', 'San Diego',
        'Houston', 'Dallas', 'Austin',
        'New York City', 'Buffalo', 'Albany',
        'Miami', 'Orlando', 'Tampa'
    ],
    'city_id': [101, 102, 103, 201, 202, 203, 301, 302, 303, 401, 402, 403],
    'state_id': [1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4]
})

print(f"Cities table shape: {cities.shape}")
cities

Cities table shape: (12, 3)


,city_name,city_id,state_id
0,Los Angeles,101,1
1,San Francisco,102,1
2,San Diego,103,1
3,Houston,201,2
4,Dallas,202,2
5,Austin,203,2
6,New York City,301,3
7,Buffalo,302,3
8,Albany,303,3
9,Miami,401,4


## 4. Define Remote Join Function

In [15]:
@ray.remote
def join_partition(cities_partition, states_table):
    """Perform a join on a partition with the broadcast states table."""
    result = cities_partition.merge(states_table, on='state_id', how='inner')
    return result

## 5. Prepare for Broadcast Join

Broadcast the small states table to all workers and partition the cities table.

In [16]:
# Put the small states table in Ray's object store (broadcast)
states_ref = ray.put(states)
print("States table broadcasted to Ray object store\n")

# Split cities table into partitions
num_partitions = 4
city_partitions = []
partition_size = len(cities) // num_partitions

for i in range(num_partitions):
    start_idx = i * partition_size
    if i == num_partitions - 1:
        end_idx = len(cities)
    else:
        end_idx = (i + 1) * partition_size
    
    partition = cities.iloc[start_idx:end_idx]
    city_partitions.append(partition)

print(f"Cities table split into {num_partitions} partitions")
print(f"Partition sizes: {[len(p) for p in city_partitions]}\n")

# Display each partition
for i, partition in enumerate(city_partitions):
    print(f"Partition {i}:")
    print(partition)
    print()

States table broadcasted to Ray object store

Cities table split into 4 partitions
Partition sizes: [3, 3, 3, 3]

Partition 0:
       city_name  city_id  state_id
0    Los Angeles      101         1
1  San Francisco      102         1
2      San Diego      103         1

Partition 1:
  city_name  city_id  state_id
3   Houston      201         2
4    Dallas      202         2
5    Austin      203         2

Partition 2:
       city_name  city_id  state_id
6  New York City      301         3
7        Buffalo      302         3
8         Albany      303         3

Partition 3:
   city_name  city_id  state_id
9      Miami      401         4
10   Orlando      402         4
11     Tampa      403         4



## 6. Execute Parallel Join

Submit join tasks to Ray workers and gather results.

In [17]:
# Perform join on each partition in parallel
print(f"Submitting {num_partitions} join tasks to Ray workers...")
futures = [join_partition.remote(partition, states_ref) for partition in city_partitions]

# Gather results
print("Gathering results...\n")
results = ray.get(futures)

# Display each joined result
for i, result_partition in enumerate(results):
    print(f"Joined result from partition {i}:")
    print(result_partition)
    print()

# Concatenate all results
result = pd.concat(results, ignore_index=True)

print(f"Join complete! Final result shape: {result.shape}")
result

Submitting 4 join tasks to Ray workers...
Gathering results...

Joined result from partition 0:
       city_name  city_id  state_id  state_name
0    Los Angeles      101         1  California
1  San Francisco      102         1  California
2      San Diego      103         1  California

Joined result from partition 1:
  city_name  city_id  state_id state_name
0   Houston      201         2      Texas
1    Dallas      202         2      Texas
2    Austin      203         2      Texas

Joined result from partition 2:
       city_name  city_id  state_id state_name
0  New York City      301         3   New York
1        Buffalo      302         3   New York
2         Albany      303         3   New York

Joined result from partition 3:
  city_name  city_id  state_id state_name
0     Miami      401         4    Florida
1   Orlando      402         4    Florida
2     Tampa      403         4    Florida

Join complete! Final result shape: (12, 4)


,city_name,city_id,state_id,state_name
0,Los Angeles,101,1,California
1,San Francisco,102,1,California
2,San Diego,103,1,California
3,Houston,201,2,Texas
4,Dallas,202,2,Texas
5,Austin,203,2,Texas
6,New York City,301,3,New York
7,Buffalo,302,3,New York
8,Albany,303,3,New York
9,Miami,401,4,Florida
